# Lesson 07 - 規劃設計模式

此筆記本展示了使用 Microsoft Agent Framework 的 AI 代理的**規劃設計模式**。  
您將學習如何將複雜的旅遊請求拆解為結構化的子任務，將其分配給專家代理，  
並執行最終規劃 — 全部使用由 Pydantic 模型支援的結構化輸出。


## 安裝設定


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 任務分解

任務分解是規劃設計模式的核心。與其讓單一代理人處理複雜請求的整個流程，我們將問題分解為較小、定義明確的**子任務**。每個子任務會分配給專門的代理人（例如，航班、飯店、活動），並設定清晰的優先順序和依賴關係。

此方法帶來幾項好處：
- **清晰度**：每個子任務有單一職責。
- **並行性**：獨立的子任務可同時執行。
- **可靠性**：故障會限制在個別子任務中。
- **預算追蹤**：費用會針對子任務估算並彙總。


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## 建立具有結構化輸出的規劃代理

規劃代理扮演著**前台協調員**的角色。根據高階的旅行需求，它會產生一個結構化的 `TravelPlan` — 將需求分解為子任務、設定優先順序，並識別相依性，以便禮賓或執行層可以執行工作。


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## 使用專家工具執行計劃

前台代理產生結構化計劃後，**禮賓代理**會執行該計劃。  
每個專家工具負責處理一類子任務（航班、飯店、活動）。禮賓代理會依照計劃中子任務的依賴順序逐一處理，並將每個子任務指派給適當的工具。


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## 總結

在本課程中，您學習了 AI 代理的**規劃設計模式**：

1. **任務分解** — 前台規劃代理使用 Pydantic 模型將複雜的旅遊請求拆解成結構化的子任務，並根據優先順序和依賴性將每個子任務分配給專家代理。
2. **結構化輸出** — 代理藉由傳遞 `response_format` 回傳經驗證的 `TravelPlan` 物件，而非自由格式文字，使後續處理更可靠。
3. **計劃執行** — 禮賓代理使用專家工具（`book_flight`、`reserve_hotel`、`book_activity`）迭代子任務以執行計劃並回報結果。

此模式將「做什麼」（規劃）與「怎麼做」（執行）分離，使代理更具模組化、易於測試且擴充性更佳。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯內容所導致的任何誤解或誤釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
